<a href="https://colab.research.google.com/gist/arulagrawal/3e79086b3bfefd462618c9c76da6e092/ablation_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RelevanceCAT Ablation: Where Do the Gains Come From?

This notebook runs a controlled comparison to show that SPLADECAT's improvement stems from **injecting SPLADE scores into the cross-encoder**, not simply from SPLADE being a stronger first-stage retriever.

**Training parity.** All cross-encoders that we compare directly (vanilla, BM25CAT, SPLADECAT) are fine-tuned on the exact same 10% slice of teacher triples with identical hyperparameters. We keep the pre-trained MiniLM cross-encoder and the first-stage retrievers in the table as reference context.

## Experimental Setup (TREC DL'19)

| # | System | Description |
|---|--------|-------------|
| 1 | BM25 (first-stage) | Raw BM25 retrieval — no re-ranking |
| 2 | SPLADE (first-stage) | Raw SPLADE retrieval — no re-ranking |
| 3 | Cross-Encoder (pre-trained, ref) | Off-the-shelf HF MiniLM cross-encoder re-ranking BM25 top-1000 |
| 4 | Cross-Encoder (10% vanilla) | MiniLM fine-tuned on 10% triples without score injection |
| 5 | Cross-Encoder + BM25CAT | Same 10% fine-tune with BM25 score injected as text |
| 6 | Cross-Encoder + SPLADECAT | Same 10% fine-tune with SPLADE score injected as text |

Metrics: **nDCG@10, MAP@1000, Recall@10**

If SPLADECAT >> SPLADE first-stage, the injection mechanism adds value beyond SPLADE retrieval quality.

## Cell 1: Install Dependencies

In [ ]:
!pip install -q "pillow<12" torch transformers sentence-transformers pyserini pytrec-eval-terrier faiss-cpu numpy tqdm pandas datasets

In [ ]:
%%capture output
import os

!apt-get install openjdk-21-jre-headless -qq > /dev/null
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
!update-alternatives --set java /usr/lib/jvm/java-21-openjdk-amd64/bin/java
!java -version

!pip install pyserini

## Cell 2: Imports & Config

In [ ]:
import os
import json
import gzip
import tarfile
import gc
import csv
import logging
import multiprocessing
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import tqdm.auto as tqdm
import pytrec_eval
from torch import nn
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoModelForMaskedLM, AutoTokenizer, AutoConfig
from sentence_transformers import util

logging.basicConfig(
    format='%(asctime)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    level=logging.INFO,
)

device = torch.device("cuda")
print(f"Device: {device} — {torch.cuda.get_device_name(0)}")

# ── Paths ──
DATA_DIR = "msmarco-data"
SCORE_DIR = "score_files"
MODEL_DIR = "finetuned_CEs"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(SCORE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Normalization constants ──
BM25_MIN, BM25_MAX = 0, 50
SPLADE_MIN, SPLADE_MAX = 0, 125807  # P99 of training distribution

# ── Model config ──
BASE_MODEL = "microsoft/MiniLM-L12-H384-uncased"
BASELINE_CE_MODEL = "cross-encoder/ms-marco-MiniLM-L-12-v2"  # pre-trained HF baseline
MAX_LENGTH_QUERY = 30
MAX_LENGTH_PASSAGE = 200
MAX_LENGTH_BASELINE = 230 + 3       # [CLS] query [SEP] doc [SEP]
MAX_LENGTH_INJECTED = 230 + 3 + 3   # extra tokens for "{score} [SEP]"

# Judged query IDs are derived from qrels (loaded in Cell 5)
JUDGED_QIDS = None  # set after loading qrels

# Store all results here
ALL_RESULTS = {}

Device: cuda — NVIDIA RTX PRO 6000 Blackwell Server Edition


## Cell 3: Download TREC DL'19 Evaluation Data

In [ ]:
# Top-1000 BM25 candidates
top1000_path = os.path.join(DATA_DIR, "msmarco-passagetest2019-top1000.tsv")
if not os.path.exists(top1000_path):
    gz_path = top1000_path + ".gz"
    if not os.path.exists(gz_path):
        print("Downloading msmarco-passagetest2019-top1000.tsv.gz...")
        util.http_get("https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-passagetest2019-top1000.tsv.gz", gz_path)
    with gzip.open(gz_path, 'rb') as f_in, open(top1000_path, 'wb') as f_out:
        f_out.write(f_in.read())

# Test queries
queries_path = os.path.join(DATA_DIR, "msmarco-test2019-queries.tsv")
if not os.path.exists(queries_path):
    gz_path = queries_path + ".gz"
    if not os.path.exists(gz_path):
        print("Downloading msmarco-test2019-queries.tsv.gz...")
        util.http_get("https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz", gz_path)
    with gzip.open(gz_path, 'rb') as f_in, open(queries_path, 'wb') as f_out:
        f_out.write(f_in.read())

# Qrels
qrel_path = os.path.join(DATA_DIR, "2019qrels-pass.txt")
if not os.path.exists(qrel_path):
    print("Downloading 2019qrels-pass.txt...")
    util.http_get("https://trec.nist.gov/data/deep/2019qrels-pass.txt", qrel_path)

# Corpus
collection_path = os.path.join(DATA_DIR, "collection.tsv")
if not os.path.exists(collection_path):
    tar_path = os.path.join(DATA_DIR, "collection.tar.gz")
    if not os.path.exists(tar_path):
        print("Downloading collection.tar.gz...")
        util.http_get("https://msmarco.z22.web.core.windows.net/msmarcoranking/collection.tar.gz", tar_path)
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=DATA_DIR)

print("All evaluation data ready.")

All evaluation data ready.


## Cell 4: Utility Code

In [ ]:
# ── CrossEncoder (CUDA-only, simplified from eval scripts) ──

class CrossEncoder:
    def __init__(self, model_name, num_labels=1, max_length=None):
        self.config = AutoConfig.from_pretrained(model_name)
        if num_labels is not None:
            self.config.num_labels = num_labels
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name, config=self.config, ignore_mismatched_sizes=True
        ).cuda().eval()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.max_length = max_length
        self.default_activation_function = nn.Sigmoid() if self.config.num_labels == 1 else nn.Identity()

    def predict(self, sentences, batch_size=64):
        all_scores = []
        for i in range(0, len(sentences), batch_size):
            batch = sentences[i:i + batch_size]
            texts_a = [s[0].strip() for s in batch]
            texts_b = [s[1].strip() for s in batch]
            tokenized = self.tokenizer(
                texts_a, texts_b,
                padding=True, truncation='longest_first',
                return_tensors='pt', max_length=self.max_length,
            )
            tokenized = {k: v.cuda() for k, v in tokenized.items()}
            with torch.no_grad():
                logits = self.model(**tokenized, return_dict=True).logits
                logits = self.default_activation_function(logits)
            if self.config.num_labels == 1:
                all_scores.extend(logits[:, 0].cpu().numpy().tolist())
            else:
                all_scores.extend(logits.cpu().numpy().tolist())
        return np.array(all_scores)

    def save(self, path):
        os.makedirs(path, exist_ok=True)
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)


# ── Data loading helpers ──

def read_collection(f_path):
    data = {}
    with open(f_path, 'r', encoding='utf8') as fp:
        for line in tqdm.tqdm(fp, desc=f"Reading {os.path.basename(f_path)}"):
            parts = line.strip().split("\t")
            if len(parts) == 2:
                data[parts[0]] = parts[1]
    return data


def truncate_queries(queries_dict, tokenizer, max_length):
    for qid, text in tqdm.tqdm(queries_dict.items(), desc="Truncating queries"):
        encoded = tokenizer(text, truncation=True, max_length=max_length, return_tensors='pt')['input_ids']
        queries_dict[qid] = tokenizer.decode(encoded[0], skip_special_tokens=True)
    return queries_dict


def evaluate_run(run, qrel, metrics={"ndcg_cut.10", "map_cut.1000", "recall.10"}):
    """Evaluate a run dict {qid: {did: score}} against qrels using pytrec_eval."""
    evaluator = pytrec_eval.RelevanceEvaluator(qrel, metrics)
    scores = evaluator.evaluate(run)
    results = {}
    for metric in metrics:
        key = metric.replace('.', '_')
        results[metric] = np.mean([s[key] for s in scores.values()])
    return results


def normalize_scores(scores_dict, min_val, max_val):
    """Normalize scores to 0-100 integer range."""
    normalized = {}
    for qid in scores_dict:
        normalized[qid] = {}
        for did, score in scores_dict[qid].items():
            n = (score - min_val) / (max_val - min_val)
            normalized[qid][did] = int(n * 100)
    return normalized


print("Utilities ready.")

Utilities ready.


## Cell 5: Load Shared Data

In [ ]:
# Load qrels — judged query IDs are derived from this
with open(qrel_path, 'r') as f:
    qrel = pytrec_eval.parse_qrel(f)

JUDGED_QIDS = set(qrel.keys())
print(f"Judged queries from qrels: {len(JUDGED_QIDS)}")

# Load corpus and queries
corpus = read_collection(collection_path)
queries = read_collection(queries_path)

# Truncate queries to max_length_query tokens
tokenizer_trunc = AutoTokenizer.from_pretrained(BASE_MODEL, truncation_side='right')
queries = truncate_queries(queries, tokenizer_trunc, MAX_LENGTH_QUERY)
del tokenizer_trunc

# Parse top-1000 file: {qid: {did: passage_text, ...}}
top1000_data = {}  # {qid: [(did, query_text, passage_text), ...]}
with open(top1000_path, 'r', encoding='utf8') as fp:
    for line in tqdm.tqdm(fp, desc="Parsing top-1000"):
        qid, did, query_text, passage = line.strip().split("\t")
        if qid not in queries:
            continue
        if qid not in JUDGED_QIDS:
            continue
        if qid not in top1000_data:
            top1000_data[qid] = []
        top1000_data[qid].append((did, corpus.get(did, passage)))

print(f"Loaded {len(corpus):,} passages, {len(queries)} queries, {len(top1000_data)} judged queries")

Judged queries from qrels: 43


Reading collection.tsv: 0it [00:00, ?it/s]

Reading msmarco-test2019-queries.tsv: 0it [00:00, ?it/s]

Truncating queries:   0%|          | 0/200 [00:00<?, ?it/s]

Parsing top-1000: 0it [00:00, ?it/s]

Loaded 8,841,823 passages, 200 queries, 43 judged queries


## Cell 6: Compute BM25 Scores for TREC'19 Top-1000

In [ ]:
bm25_scores_path = os.path.join(SCORE_DIR, "3_trec19_bm25_scores.json")

if os.path.exists(bm25_scores_path):
    print(f"Loading cached BM25 scores from {bm25_scores_path}")
    with open(bm25_scores_path, 'r') as f:
        bm25_scores = json.load(f)
else:
    from pyserini.index.lucene import LuceneIndexReader as IndexReader
    from pyserini.pyclass import autoclass

    index_reader = IndexReader.from_prebuilt_index('msmarco-v1-passage')
    similarity_bm25 = autoclass('org.apache.lucene.search.similarities.BM25Similarity')(0.82, 0.68)

    bm25_scores = {}
    for qid, docs in tqdm.tqdm(top1000_data.items(), desc="Computing BM25 scores"):
        query = queries[qid]
        bm25_scores[qid] = {}
        for did, _ in docs:
            score = index_reader.compute_query_document_score(did, query, similarity=similarity_bm25)
            bm25_scores[qid][did] = score

    with open(bm25_scores_path, 'w') as f:
        json.dump(bm25_scores, f)
    print(f"Saved BM25 scores to {bm25_scores_path}")

print(f"BM25 scores: {len(bm25_scores)} queries")

Loading cached BM25 scores from score_files/3_trec19_bm25_scores.json
BM25 scores: 43 queries


## Cell 7: Evaluate BM25 First-Stage

In [ ]:
# Build run from raw BM25 scores
bm25_run = {}
for qid in bm25_scores:
    if qid in JUDGED_QIDS:
        bm25_run[qid] = {did: float(score) for did, score in bm25_scores[qid].items()}

ALL_RESULTS['BM25 (first-stage)'] = evaluate_run(bm25_run, qrel)
print("BM25 first-stage:", ALL_RESULTS['BM25 (first-stage)'])

BM25 first-stage: {'recall.10': np.float64(0.1385199640810133), 'ndcg_cut.10': np.float64(0.5203010626017599), 'map_cut.1000': np.float64(0.3719108990277008)}


## Cell 8: Compute SPLADE Scores for TREC'19 Top-1000

In [ ]:
splade_scores_path = os.path.join(SCORE_DIR, "3_trec19_splade_scores.json")

if os.path.exists(splade_scores_path):
    print(f"Loading cached SPLADE scores from {splade_scores_path}")
    with open(splade_scores_path, 'r') as f:
        splade_scores = json.load(f)
else:
    from pyserini.search.lucene import LuceneImpactSearcher

    print("Loading pre-built SPLADE index...")
    searcher = LuceneImpactSearcher.from_prebuilt_index(
        'msmarco-v1-passage.splade-pp-ed',
        'naver/splade-cocondenser-ensembledistil'
    )

    # Collect all unique queries and their candidate docs from top-1000
    all_query_texts = {}
    all_query_docs = {}
    with open(top1000_path, 'r', encoding='utf8') as fp:
        for line in fp:
            qid, did, query_text, passage = line.strip().split("\t")
            all_query_texts[qid] = query_text
            if qid not in all_query_docs:
                all_query_docs[qid] = set()
            all_query_docs[qid].add(did)

    splade_scores = {}
    for qid, query_text in tqdm.tqdm(all_query_texts.items(), desc="SPLADE scoring"):
        hits = searcher.search(query_text, k=1000)
        hit_scores = {hit.docid: hit.score for hit in hits}
        splade_scores[qid] = {}
        for did in all_query_docs[qid]:
            splade_scores[qid][did] = float(hit_scores.get(did, 0.0))

    del searcher
    with open(splade_scores_path, 'w') as f:
        json.dump(splade_scores, f)
    print(f"Saved SPLADE scores to {splade_scores_path}")

print(f"SPLADE scores: {len(splade_scores)} queries")

Loading cached SPLADE scores from score_files/3_trec19_splade_scores.json
SPLADE scores: 200 queries


## Cell 9: Evaluate SPLADE First-Stage

In [ ]:
# SPLADE first-stage: use SPLADE's own retrieval ranking
# We search the full index and evaluate the SPLADE ranking directly.
from pyserini.search.lucene import LuceneImpactSearcher

print("Loading SPLADE index for first-stage evaluation...")
splade_searcher = LuceneImpactSearcher.from_prebuilt_index(
    'msmarco-v1-passage.splade-pp-ed',
    'naver/splade-cocondenser-ensembledistil'
)

# Load original (un-truncated) test queries for SPLADE retrieval
orig_queries = read_collection(queries_path)

splade_run = {}
for qid in tqdm.tqdm(JUDGED_QIDS, desc="SPLADE retrieval"):
    if qid not in orig_queries:
        continue
    hits = splade_searcher.search(orig_queries[qid], k=1000)
    splade_run[qid] = {hit.docid: float(hit.score) for hit in hits}

del splade_searcher

ALL_RESULTS['SPLADE (first-stage)'] = evaluate_run(splade_run, qrel)
print("SPLADE first-stage:", ALL_RESULTS['SPLADE (first-stage)'])

Loading SPLADE index for first-stage evaluation...
Attempting to initialize prebuilt index msmarco-v1-passage.splade-pp-ed.
/root/.cache/pyserini/indexes/lucene-inverted.msmarco-v1-passage.splade-pp-ed.20230524.a59610.2c008fc36131e27966a72292932358e6 already exists, skipping download.
Initializing msmarco-v1-passage.splade-pp-ed...


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: naver/splade-cocondenser-ensembledistil
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reading msmarco-test2019-queries.tsv: 0it [00:00, ?it/s]

SPLADE retrieval:   0%|          | 0/43 [00:00<?, ?it/s]

SPLADE first-stage: {'recall.10': np.float64(0.17242267763580152), 'ndcg_cut.10': np.float64(0.730806791804217), 'map_cut.1000': np.float64(0.525721817172327)}


## Cell 10: Download Training Data & Pre-computed Scores

In [ ]:
injection_folder = os.path.join(DATA_DIR, "injection_scores")
os.makedirs(injection_folder, exist_ok=True)

# BM25 training scores (from Dropbox)
bm25_train_scores_path = os.path.join(injection_folder, '1_bm25_scores_train_triples_small.json')
if not os.path.exists(bm25_train_scores_path):
    print("Downloading BM25 training scores...")
    util.http_get(
        'https://www.dropbox.com/scl/fi/ssgpoun44jtlrwy24wrad/1_bm25_scores_train_triples_small.json?rlkey=3og8ayxmyjxsei7okdumseaq7&raw=1',
        bm25_train_scores_path
    )

bm25_val_scores_path = os.path.join(injection_folder, '5_bm25_scores_train-eval_triples.json')
if not os.path.exists(bm25_val_scores_path):
    print("Downloading BM25 validation scores...")
    util.http_get(
        'https://www.dropbox.com/scl/fi/q433llwfdk701x336ce3p/5_bm25_scores_train-eval_triples.json?rlkey=5782bylutyzmk10f1uax3iao5&raw=1',
        bm25_val_scores_path
    )

# Training queries
train_queries_path = os.path.join(DATA_DIR, 'queries.train.tsv')
if not os.path.exists(train_queries_path):
    tar_path = os.path.join(DATA_DIR, 'queries.tar.gz')
    if not os.path.exists(tar_path):
        print("Downloading queries.tar.gz...")
        util.http_get('https://msmarco.z22.web.core.windows.net/msmarcoranking/queries.tar.gz', tar_path)
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(path=DATA_DIR)

# Validation triples
train_eval_path = os.path.join(DATA_DIR, 'msmarco-qidpidtriples.rnd-shuf.train-eval.tsv.gz')
if not os.path.exists(train_eval_path):
    print("Downloading validation triples...")
    util.http_get('https://sbert.net/datasets/msmarco-qidpidtriples.rnd-shuf.train-eval.tsv.gz', train_eval_path)

# Teacher logits
teacher_logits_path = os.path.join(DATA_DIR, 'bert_cat_ensemble_msmarcopassage_train_scores_ids.tsv')
if not os.path.exists(teacher_logits_path):
    print("Downloading teacher logits from HuggingFace...")
    from datasets import load_dataset as _load_dataset

    _mse_ds = _load_dataset("sentence-transformers/msmarco", "bert-ensemble-mse", split="train")
    _score_lookup = {}
    for row in tqdm.tqdm(_mse_ds, desc="Building score lookup"):
        qid = str(row['query_id'])
        if qid not in _score_lookup:
            _score_lookup[qid] = {}
        _score_lookup[qid][str(row['passage_id'])] = row['score']
    del _mse_ds; gc.collect()

    _margin_ds = _load_dataset("sentence-transformers/msmarco", "bert-ensemble-margin-mse", split="train")
    with open(teacher_logits_path, 'w', encoding='utf8') as fOut:
        for row in tqdm.tqdm(_margin_ds, desc="Writing teacher logits TSV"):
            qid = str(row['query_id'])
            pos_id = str(row['positive_id'])
            neg_id = str(row['negative_id'])
            pos_score = _score_lookup.get(qid, {}).get(pos_id, 0.0)
            neg_score = _score_lookup.get(qid, {}).get(neg_id, 0.0)
            fOut.write(f"{pos_score}\t{neg_score}\t{qid}\t{pos_id}\t{neg_id}\n")
    del _margin_ds, _score_lookup; gc.collect()

print("All training data ready.")

Building score lookup:   0%|          | 0/79561408 [00:00<?, ?it/s]

bert-ensemble-margin-mse/train-00000-of-(…):   0%|          | 0.00/281M [00:00<?, ?B/s]

bert-ensemble-margin-mse/train-00001-of-(…):   0%|          | 0.00/282M [00:00<?, ?B/s]

bert-ensemble-margin-mse/train-00002-of-(…):   0%|          | 0.00/281M [00:00<?, ?B/s]

bert-ensemble-margin-mse/train-00003-of-(…):   0%|          | 0.00/281M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/39780704 [00:00<?, ? examples/s]

Writing teacher logits TSV:   0%|          | 0/39780704 [00:00<?, ?it/s]

All training data ready.


## Cell 11: Compute SPLADE Training & Validation Scores (GPU)

In [ ]:
splade_train_scores_path = os.path.join(SCORE_DIR, "1_splade_scores_train_triples_small_gpu.json")
splade_val_scores_path = os.path.join(SCORE_DIR, "5_splade_scores_train-eval_triples.json")

if os.path.exists(splade_train_scores_path) and os.path.exists(splade_val_scores_path):
    print("SPLADE training & validation scores already exist. Skipping.")
else:
    from pyserini.search.lucene import LuceneImpactSearcher

    # ── Load training queries ──
    train_queries = read_collection(train_queries_path)

    # ── Parse training triples to find query-doc pairs ──
    print("Parsing training triples...")
    train_qd_pairs = {}  # {qid: set(dids)}
    with open(teacher_logits_path, 'rt') as fIn:
        for line in tqdm.tqdm(fIn, unit_scale=True, desc="Reading triples"):
            pos_score, neg_score, qid, pos_id, neg_id = line.strip().split("\t")
            if qid not in train_qd_pairs:
                train_qd_pairs[qid] = set()
            train_qd_pairs[qid].add(pos_id)
            train_qd_pairs[qid].add(neg_id)

    # ── Parse validation triples ──
    print("Parsing validation triples...")
    val_qd_pairs = {}
    with gzip.open(train_eval_path, 'rt') as fIn:
        for line in fIn:
            qid, pos_id, neg_id = line.strip().split()
            if qid not in val_qd_pairs:
                val_qd_pairs[qid] = set()
            val_qd_pairs[qid].add(pos_id)
            val_qd_pairs[qid].add(neg_id)

    # ── Phase 1: Batch-encode queries on GPU ──
    all_qids = list(set(list(train_qd_pairs.keys()) + list(val_qd_pairs.keys())))
    encoded_queries_path = os.path.join(SCORE_DIR, "splade_encoded_queries.json")

    if os.path.exists(encoded_queries_path):
        print(f"Loading cached encoded queries from {encoded_queries_path}")
        with open(encoded_queries_path, 'r') as f:
            encoded_queries = json.load(f)
    else:
        SPLADE_MODEL = "naver/splade-cocondenser-ensembledistil"
        splade_tokenizer = AutoTokenizer.from_pretrained(SPLADE_MODEL)
        splade_model = AutoModelForMaskedLM.from_pretrained(SPLADE_MODEL).cuda().eval()
        reverse_vocab = {v: k for k, v in splade_tokenizer.vocab.items()}

        WEIGHT_RANGE = 5
        QUANT_RANGE = 256
        BATCH_SIZE = 128
        ENCODE_CHUNK = 50000

        partial_path = encoded_queries_path + ".partial"
        encoded_queries = {}
        if os.path.exists(partial_path):
            with open(partial_path, 'r') as f:
                encoded_queries = json.load(f)
            print(f"Resuming with {len(encoded_queries)} queries already encoded")

        remaining = [qid for qid in all_qids if qid not in encoded_queries]
        print(f"Encoding {len(remaining)} queries on GPU...")

        for chunk_start in tqdm.trange(0, len(remaining), ENCODE_CHUNK, desc="Phase 1: encoding"):
            chunk_qids = remaining[chunk_start:chunk_start + ENCODE_CHUNK]
            chunk_texts = [train_queries[qid] for qid in chunk_qids]

            for batch_start in range(0, len(chunk_texts), BATCH_SIZE):
                batch_texts = chunk_texts[batch_start:batch_start + BATCH_SIZE]
                batch_qids = chunk_qids[batch_start:batch_start + BATCH_SIZE]

                tokens = splade_tokenizer(
                    batch_texts, return_tensors='pt', truncation=True,
                    max_length=64, padding=True, add_special_tokens=True,
                ).to('cuda')

                with torch.no_grad():
                    logits = splade_model(**tokens)['logits']

                reps = torch.max(
                    torch.log(1 + torch.relu(logits)) * tokens['attention_mask'].unsqueeze(-1),
                    dim=1,
                )[0].cpu().numpy()

                for i, qid in enumerate(batch_qids):
                    nonzero = np.nonzero(reps[i])[0]
                    weights = reps[i][nonzero]
                    encoded_queries[qid] = {
                        reverse_vocab[int(idx)]: int(round(float(w) / WEIGHT_RANGE * QUANT_RANGE))
                        for idx, w in zip(nonzero, weights)
                    }

            with open(partial_path, 'w') as f:
                json.dump(encoded_queries, f)

        with open(encoded_queries_path, 'w') as f:
            json.dump(encoded_queries, f)
        if os.path.exists(partial_path):
            os.remove(partial_path)

        del splade_model, splade_tokenizer
        torch.cuda.empty_cache()
        print(f"Phase 1 done: encoded {len(encoded_queries)} queries")

    # ── Phase 2: Batch search pre-built index ──
    print("Phase 2: Loading SPLADE index for batch search...")
    searcher = LuceneImpactSearcher.from_prebuilt_index(
        'msmarco-v1-passage.splade-pp-ed',
        'naver/splade-cocondenser-ensembledistil'
    )
    min_idf = searcher.min_idf
    idf = searcher.idf
    threads = multiprocessing.cpu_count()

    from pyserini.pyclass import autoclass
    JHashMap = autoclass('java.util.HashMap')
    JInt = autoclass('java.lang.Integer')
    JArrayList = autoclass('java.util.ArrayList')
    JString = autoclass('java.lang.String')

    SEARCH_CHUNK = 10000

    def batch_search_splade(qd_pairs, output_path):
        """Search SPLADE index for all query-doc pairs, save results."""
        partial = output_path + ".partial"
        scores_dict = {}
        if os.path.exists(partial):
            with open(partial, 'r') as f:
                scores_dict = json.load(f)

        unique_qids = list(qd_pairs.keys())
        remaining = [qid for qid in unique_qids if qid not in scores_dict]
        print(f"  Searching {len(remaining)} queries...")

        for chunk_start in tqdm.trange(0, len(remaining), SEARCH_CHUNK, desc="Batch search"):
            chunk_qids = remaining[chunk_start:chunk_start + SEARCH_CHUNK]

            query_lst = JArrayList()
            qid_lst = JArrayList()
            for qid in chunk_qids:
                if qid not in encoded_queries:
                    continue
                jquery = JHashMap()
                for token, weight in encoded_queries[qid].items():
                    if token in idf and idf[token] > min_idf:
                        jquery.put(token, JInt(int(weight)))
                query_lst.add(jquery)
                qid_lst.add(JString(qid))

            results = searcher.object.batch_search(query_lst, qid_lst, 1000, threads)

            for entry in results.entrySet().toArray():
                qid = entry.getKey()
                hits = entry.getValue()
                hit_scores = {hit.docid: hit.score for hit in hits}
                scores_dict[qid] = {}
                for did in qd_pairs.get(qid, set()):
                    scores_dict[qid][did] = float(hit_scores.get(did, 0.0))

            with open(partial, 'w') as f:
                json.dump(scores_dict, f)

        with open(output_path, 'w') as f:
            json.dump(scores_dict, f)
        if os.path.exists(partial):
            os.remove(partial)
        return scores_dict

    if not os.path.exists(splade_train_scores_path):
        print("Computing SPLADE training scores...")
        batch_search_splade(train_qd_pairs, splade_train_scores_path)

    if not os.path.exists(splade_val_scores_path):
        print("Computing SPLADE validation scores...")
        batch_search_splade(val_qd_pairs, splade_val_scores_path)

    del searcher
    gc.collect()

print("SPLADE training & validation scores ready.")

Reading queries.train.tsv: 0it [00:00, ?it/s]

Parsing training triples...


Reading triples: 0.00it [00:00, ?it/s]

Parsing validation triples...


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

...

Computing SPLADE training scores...
  Searching 398791 queries...


Batch search:   0%|          | 0/40 [00:00<?, ?it/s]

Computing SPLADE validation scores...
  Searching 500 queries...


Batch search:   0%|          | 0/1 [00:00<?, ?it/s]

SPLADE training & validation scores ready.


## Cell 12: Train Vanilla Cross-Encoder (10% data, no injection)

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder as STCrossEncoder
from sentence_transformers.cross_encoder.trainer import CrossEncoderTrainer
from sentence_transformers.cross_encoder.training_args import CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.losses import MSELoss as CEMSELoss
from datasets import Dataset, Features, Value
VANILLA_SAVE_PATH = os.path.join(MODEL_DIR, 'vanilla-' + datetime.now().strftime('%Y%m%d_%H%M%S'))
if 'CERerankingEvaluator' not in globals():
    class CERerankingEvaluator:
        def __init__(self, samples, mrr_at_k=10, name='', write_csv=True):
            self.samples = list(samples.values()) if isinstance(samples, dict) else samples
            self.mrr_at_k = mrr_at_k
            self.name = name
            self.csv_file = f"CERerankingEvaluator_{name}_results.csv" if name else "CERerankingEvaluator_results.csv"
            self.csv_headers = ["epoch", "steps", f"MRR@{mrr_at_k}"]
            self.write_csv = write_csv
        def __call__(self, model, output_path=None, epoch=-1, steps=-1):
            all_mrr = []
            for instance in self.samples:
                queries_list = instance['query']
                positive = list(instance['positive'])
                negative = list(instance['negative'])
                docs = positive + negative
                is_relevant = [True] * len(positive) + [False] * len(negative)
                if not positive or not negative:
                    continue
                model_input = [[q, d] for q, d in zip(queries_list, docs)]
                pred_scores = model.predict(model_input, convert_to_numpy=True, show_progress_bar=False)
                pred_scores_argsort = np.argsort(-pred_scores)
                mrr_score = 0
                for rank, index in enumerate(pred_scores_argsort[:self.mrr_at_k]):
                    if is_relevant[index]:
                        mrr_score = 1 / (rank + 1)
                        break
                all_mrr.append(mrr_score)
            mean_mrr = np.mean(all_mrr)
            print(f"  MRR@{self.mrr_at_k}: {mean_mrr*100:.2f}")
            if output_path and self.write_csv:
                csv_path = os.path.join(output_path, self.csv_file)
                exists = os.path.isfile(csv_path)
                with open(csv_path, 'a' if exists else 'w', encoding='utf-8') as f:
                    writer = csv.writer(f)
                    if not exists:
                        writer.writerow(self.csv_headers)
                    writer.writerow([epoch, steps, mean_mrr])
            return mean_mrr
def _dedup_tsv(input_path, output_path):
    seen = set()
    kept = 0
    with open(input_path, 'r', encoding='utf8') as f_in, open(output_path, 'w', encoding='utf8') as f_out:
        for line in f_in:
            line = line.strip()
            if not line:
                continue
            if line in seen:
                continue
            seen.add(line)
            f_out.write(line + "\n")
            kept += 1
    return kept
print("Building dev set for vanilla cross-encoder...")
train_corpus = read_collection(collection_path)
train_queries = read_collection(train_queries_path)
dev_samples = {}
with gzip.open(train_eval_path, 'rt', encoding='utf8') as fIn:
    for line in fIn:
        qid, pos_id, neg_id = line.strip().split()
        if qid not in train_queries or pos_id not in train_corpus or neg_id not in train_corpus:
            continue
        if qid not in dev_samples and len(dev_samples) < 200:
            dev_samples[qid] = {'query': [], 'positive': [], 'negative': []}
        if qid in dev_samples:
            dev_samples[qid]['positive'].append(train_corpus[pos_id])
            dev_samples[qid]['query'].append(train_queries[qid])
            if len(dev_samples[qid]['negative']) < 200:
                dev_samples[qid]['negative'].append(train_corpus[neg_id])
                dev_samples[qid]['query'].append(train_queries[qid])
dev_qids = set(dev_samples.keys())
print(f"Dev set: {len(dev_qids)} queries")
# ── Load score lookups for consistent triple filtering ──
# All three models (vanilla, BM25CAT, SPLADECAT) must train on the exact same
# triples.  BM25CAT and SPLADECAT skip any triple whose score is missing, so
# vanilla must apply the identical filter.
print("Loading score files for consistent filtering...")
with open(bm25_train_scores_path, 'r') as f:
    _bm25_train = json.load(f)
with open(bm25_val_scores_path, 'r') as f:
    _bm25_val = json.load(f)
_bm25_lookup = {}
for src in [_bm25_train, _bm25_val]:
    for qid in src:
        if qid not in _bm25_lookup:
            _bm25_lookup[qid] = set()
        for did in src[qid]:
            _bm25_lookup[qid].add(did)
del _bm25_train, _bm25_val
with open(splade_train_scores_path, 'r') as f:
    _splade_train = json.load(f)
with open(splade_val_scores_path, 'r') as f:
    _splade_val = json.load(f)
_splade_lookup = {}
for src in [_splade_train, _splade_val]:
    for qid in src:
        if qid not in _splade_lookup:
            _splade_lookup[qid] = set()
        for did in src[qid]:
            _splade_lookup[qid].add(did)
del _splade_train, _splade_val
SAMPLE_EVERY_N = 10
vanilla_train_data_path = os.path.join(DATA_DIR, 'vanilla_train_data_10pct.tsv')
if not os.path.exists(vanilla_train_data_path):
    print("Writing 10% vanilla training data (filtered to shared triples)...")
    num_samples = 0
    line_idx = 0
    tmp_path = vanilla_train_data_path + '.raw'
    with open(teacher_logits_path, encoding='utf8') as fIn, open(tmp_path, 'w', encoding='utf8') as fOut:
        for line in fIn:
            pos_score, neg_score, qid, pid1, pid2 = line.strip().split("\t")
            if qid in dev_qids:
                continue
            line_idx += 1
            if line_idx % SAMPLE_EVERY_N != 0:
                continue
            if qid not in train_queries or pid1 not in train_corpus or pid2 not in train_corpus:
                continue
            # Same filter as BM25CAT and SPLADECAT — keep only triples with all scores
            if qid not in _bm25_lookup or pid1 not in _bm25_lookup.get(qid, set()) or pid2 not in _bm25_lookup.get(qid, set()):
                continue
            if qid not in _splade_lookup or pid1 not in _splade_lookup.get(qid, set()) or pid2 not in _splade_lookup.get(qid, set()):
                continue
            fOut.write(f"{train_queries[qid]}\t{train_corpus[pid1]}\t{pos_score}\n")
            fOut.write(f"{train_queries[qid]}\t{train_corpus[pid2]}\t{neg_score}\n")
            num_samples += 2
    kept = _dedup_tsv(tmp_path, vanilla_train_data_path)
    os.remove(tmp_path)
    print(f"Wrote {num_samples} training samples, kept {kept} unique examples")
del _bm25_lookup, _splade_lookup; gc.collect()
features_vanilla = Features({
    'sentence1': Value('string'),
    'sentence2': Value('string'),
    'label': Value('float32'),
})
def gen_vanilla():
    with open(vanilla_train_data_path, 'r', encoding='utf8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 3:
                continue
            sentence1 = parts[0]
            label_str = parts[-1]
            sentence2 = '\t'.join(parts[1:-1]) if len(parts) > 2 else ''
            if not sentence2:
                sentence2 = ''
            try:
                label = float(label_str)
            except ValueError:
                continue
            yield {'sentence1': sentence1, 'sentence2': sentence2, 'label': label}
vanilla_dataset = Dataset.from_generator(gen_vanilla, features=features_vanilla)
print(f"Vanilla dataset: {len(vanilla_dataset)} samples")
del train_corpus, train_queries
gc.collect()
vanilla_model = STCrossEncoder(
    BASE_MODEL, num_labels=1, max_length=512,
    activation_fn=torch.nn.Identity(), device='cuda'
)
vanilla_evaluator = CERerankingEvaluator(dev_samples, name='vanilla-eval')
vanilla_args = CrossEncoderTrainingArguments(
    output_dir=VANILLA_SAVE_PATH,
    num_train_epochs=1,
    per_device_train_batch_size=256,
    warmup_steps=625,
    learning_rate=7e-6,
    bf16=True,
    eval_strategy='steps',
    eval_steps=5000,
    save_strategy='steps',
    save_steps=5000,
    load_best_model_at_end=True,
    metric_for_best_model='eval_sequential_score',
    greater_is_better=True,
    max_grad_norm=1.0,
    weight_decay=0.01,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    logging_steps=100,
)
vanilla_trainer = CrossEncoderTrainer(
    model=vanilla_model,
    args=vanilla_args,
    train_dataset=vanilla_dataset,
    loss=CEMSELoss(vanilla_model),
    evaluator=[vanilla_evaluator],
)
print("Training vanilla cross-encoder (10% data)...")
vanilla_trainer.train()
vanilla_model.save(VANILLA_SAVE_PATH + '-latest')
print(f"Vanilla model saved to {VANILLA_SAVE_PATH}-latest")
del vanilla_model, vanilla_trainer, vanilla_dataset, dev_samples
torch.cuda.empty_cache(); gc.collect()

Building dev set for vanilla cross-encoder...


Reading collection.tsv: 0it [00:00, ?it/s]

Reading queries.train.tsv: 0it [00:00, ?it/s]

Dev set: 200 queries
Loading score files for consistent filtering...
Writing 10% vanilla training data (filtered to shared triples)...
Wrote 7951856 training samples, kept 4382298 unique examples


Generating train split: 0 examples [00:00, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Vanilla model saved to finetuned_CEs/vanilla-20260412_192316-latest


3724

## Cell 13: Train BM25CAT Cross-Encoder

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder as STCrossEncoder
from sentence_transformers.cross_encoder.trainer import CrossEncoderTrainer
from sentence_transformers.cross_encoder.training_args import CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.losses import MSELoss as CEMSELoss
from datasets import Dataset

BM25CAT_SAVE_PATH = os.path.join(MODEL_DIR, 'bm25cat-' + datetime.now().strftime('%Y%m%d_%H%M%S'))

# ── CERerankingEvaluator (inline, from train/CERerankingEvaluator_bm25cat.py) ──
class CERerankingEvaluator:
    def __init__(self, samples, mrr_at_k=10, name='', write_csv=True):
        self.samples = list(samples.values()) if isinstance(samples, dict) else samples
        self.mrr_at_k = mrr_at_k
        self.name = name
        self.csv_file = f"CERerankingEvaluator_{name}_results.csv" if name else "CERerankingEvaluator_results.csv"
        self.csv_headers = ["epoch", "steps", f"MRR@{mrr_at_k}"]
        self.write_csv = write_csv

    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        all_mrr = []
        for instance in self.samples:
            queries_list = instance['query']
            positive = list(instance['positive'])
            negative = list(instance['negative'])
            docs = positive + negative
            is_relevant = [True] * len(positive) + [False] * len(negative)
            if not positive or not negative:
                continue
            model_input = [[q, d] for q, d in zip(queries_list, docs)]
            pred_scores = model.predict(model_input, convert_to_numpy=True, show_progress_bar=False)
            pred_scores_argsort = np.argsort(-pred_scores)
            mrr_score = 0
            for rank, index in enumerate(pred_scores_argsort[:self.mrr_at_k]):
                if is_relevant[index]:
                    mrr_score = 1 / (rank + 1)
                    break
            all_mrr.append(mrr_score)
        mean_mrr = np.mean(all_mrr)
        print(f"  MRR@{self.mrr_at_k}: {mean_mrr*100:.2f}")
        if output_path and self.write_csv:
            csv_path = os.path.join(output_path, self.csv_file)
            exists = os.path.isfile(csv_path)
            with open(csv_path, 'a' if exists else 'w', encoding='utf-8') as f:
                writer = csv.writer(f)
                if not exists:
                    writer.writerow(self.csv_headers)
                writer.writerow([epoch, steps, mean_mrr])
        return mean_mrr


# ── Load & normalize BM25 scores ──
print("Loading BM25 training scores...")
with open(bm25_train_scores_path, 'r') as f:
    bm25_train_scores = json.load(f)
with open(bm25_val_scores_path, 'r') as f:
    bm25_val_scores = json.load(f)

# Merge and normalize
bm25_all_scores = {}
for src in [bm25_train_scores, bm25_val_scores]:
    for qid in src:
        if qid not in bm25_all_scores:
            bm25_all_scores[qid] = {}
        for did, score in src[qid].items():
            n = (score - BM25_MIN) / (BM25_MAX - BM25_MIN)
            bm25_all_scores[qid][did] = int(n * 100)

del bm25_train_scores, bm25_val_scores; gc.collect()

# ── Load corpus & queries for training ──
train_corpus = read_collection(collection_path)
train_queries = read_collection(train_queries_path)

# ── Build dev set ──
dev_samples = {}
with gzip.open(train_eval_path, 'rt', encoding='utf8') as fIn:
    for line in fIn:
        qid, pos_id, neg_id = line.strip().split()
        if qid not in train_queries or pos_id not in train_corpus or neg_id not in train_corpus:
            continue
        if qid not in bm25_all_scores or pos_id not in bm25_all_scores.get(qid, {}) or neg_id not in bm25_all_scores.get(qid, {}):
            continue
        if qid not in dev_samples and len(dev_samples) < 200:
            dev_samples[qid] = {'query': [], 'positive': [], 'negative': []}
        if qid in dev_samples:
            dev_samples[qid]['positive'].append(train_corpus[pos_id])
            dev_samples[qid]['query'].append(f"{bm25_all_scores[qid][pos_id]} [SEP] {train_queries[qid]}")
            if len(dev_samples[qid]['negative']) < 200:
                dev_samples[qid]['negative'].append(train_corpus[neg_id])
                dev_samples[qid]['query'].append(f"{bm25_all_scores[qid][neg_id]} [SEP] {train_queries[qid]}")

dev_qids = set(dev_samples.keys())
print(f"Dev set: {len(dev_qids)} queries")

# ── Write pre-processed training data ──
SAMPLE_EVERY_N = 10
bm25cat_train_data_path = os.path.join(DATA_DIR, 'bm25cat_train_data_10pct.tsv')

if not os.path.exists(bm25cat_train_data_path):
    print("Writing 10% BM25CAT training data...")
    num_samples = 0
    line_idx = 0
    with open(teacher_logits_path, encoding='utf8') as fIn, open(bm25cat_train_data_path, 'w', encoding='utf8') as fOut:
        for line in fIn:
            pos_score, neg_score, qid, pid1, pid2 = line.strip().split("\t")
            if qid in dev_qids:
                continue
            line_idx += 1
            if line_idx % SAMPLE_EVERY_N != 0:
                continue
            if qid not in train_queries or pid1 not in train_corpus or pid2 not in train_corpus:
                continue
            if qid not in bm25_all_scores or pid1 not in bm25_all_scores.get(qid, {}) or pid2 not in bm25_all_scores.get(qid, {}):
                continue
            q1 = f"{bm25_all_scores[qid][pid1]} [SEP] {train_queries[qid]}"
            q2 = f"{bm25_all_scores[qid][pid2]} [SEP] {train_queries[qid]}"
            fOut.write(f"{q1}\t{train_corpus[pid1]}\t{pos_score}\n")
            fOut.write(f"{q2}\t{train_corpus[pid2]}\t{neg_score}\n")
            num_samples += 2
    print(f"Wrote {num_samples} training samples")

# Free memory
del train_corpus, train_queries, bm25_all_scores; gc.collect()

# ── Build HF Dataset ──
def gen_bm25cat():
    with open(bm25cat_train_data_path, 'r', encoding='utf8') as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 3:
                yield {"sentence1": parts[0], "sentence2": parts[1], "label": float(parts[2])}

bm25cat_dataset = Dataset.from_generator(gen_bm25cat)
print(f"BM25CAT dataset: {len(bm25cat_dataset)} samples")

# ── Train ──
bm25cat_model = STCrossEncoder(
    BASE_MODEL, num_labels=1, max_length=512,
    activation_fn=torch.nn.Identity(), device='cuda'
)

bm25cat_evaluator = CERerankingEvaluator(dev_samples, name='bm25cat-eval')

bm25cat_args = CrossEncoderTrainingArguments(
    output_dir=BM25CAT_SAVE_PATH,
    num_train_epochs=1,
    per_device_train_batch_size=256,
    warmup_steps=625,
    learning_rate=7e-6,
    bf16=True,
    eval_strategy='steps',
    eval_steps=5000,
    save_strategy='steps',
    save_steps=5000,
    load_best_model_at_end=True,
    metric_for_best_model='eval_sequential_score',
    greater_is_better=True,
    max_grad_norm=1.0,
    weight_decay=0.01,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    logging_steps=100,
)

bm25cat_trainer = CrossEncoderTrainer(
    model=bm25cat_model,
    args=bm25cat_args,
    train_dataset=bm25cat_dataset,
    loss=CEMSELoss(bm25cat_model),
    evaluator=[bm25cat_evaluator],
)

print("Training BM25CAT...")
bm25cat_trainer.train()
bm25cat_model.save(BM25CAT_SAVE_PATH + '-latest')
print(f"BM25CAT model saved to {BM25CAT_SAVE_PATH}-latest")

del bm25cat_model, bm25cat_trainer, bm25cat_dataset, dev_samples
torch.cuda.empty_cache(); gc.collect()

Loading BM25 training scores...


Reading collection.tsv: 0it [00:00, ?it/s]

Reading queries.train.tsv: 0it [00:00, ?it/s]

Dev set: 200 queries
Writing 10% BM25CAT training data...
Wrote 7951856 training samples


Generating train split: 0 examples [00:00, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BM25CAT model saved to finetuned_CEs/bm25cat-20260412_164802-latest


3757

## Cell 14: Train SPLADECAT Cross-Encoder

In [ ]:
SPLADECAT_SAVE_PATH = os.path.join(MODEL_DIR, 'spladecat-' + datetime.now().strftime('%Y%m%d_%H%M%S'))

# ── Load & normalize SPLADE scores ──
print("Loading SPLADE training scores...")
with open(splade_train_scores_path, 'r') as f:
    splade_train_raw = json.load(f)
with open(splade_val_scores_path, 'r') as f:
    splade_val_raw = json.load(f)

splade_all_scores = {}
for src in [splade_train_raw, splade_val_raw]:
    for qid in src:
        if qid not in splade_all_scores:
            splade_all_scores[qid] = {}
        for did, score in src[qid].items():
            n = (score - SPLADE_MIN) / (SPLADE_MAX - SPLADE_MIN)
            splade_all_scores[qid][did] = int(n * 100)

del splade_train_raw, splade_val_raw; gc.collect()

# ── Load corpus & queries ──
train_corpus = read_collection(collection_path)
train_queries = read_collection(train_queries_path)

# ── Build dev set ──
dev_samples = {}
with gzip.open(train_eval_path, 'rt') as fIn:
    for line in fIn:
        qid, pos_id, neg_id = line.strip().split()
        if qid not in train_queries or pos_id not in train_corpus or neg_id not in train_corpus:
            continue
        if qid not in splade_all_scores or pos_id not in splade_all_scores.get(qid, {}) or neg_id not in splade_all_scores.get(qid, {}):
            continue
        if qid not in dev_samples and len(dev_samples) < 200:
            dev_samples[qid] = {'query': [], 'positive': [], 'negative': []}
        if qid in dev_samples:
            dev_samples[qid]['positive'].append(train_corpus[pos_id])
            dev_samples[qid]['query'].append(f"{splade_all_scores[qid][pos_id]} [SEP] {train_queries[qid]}")
            if len(dev_samples[qid]['negative']) < 200:
                dev_samples[qid]['negative'].append(train_corpus[neg_id])
                dev_samples[qid]['query'].append(f"{splade_all_scores[qid][neg_id]} [SEP] {train_queries[qid]}")

dev_qids = set(dev_samples.keys())
print(f"Dev set: {len(dev_qids)} queries")

# ── Write pre-processed training data ──
spladecat_train_data_path = os.path.join(DATA_DIR, 'spladecat_train_data_10pct.tsv')

if not os.path.exists(spladecat_train_data_path):
    print("Writing 10% SPLADECAT training data...")
    num_samples = 0
    line_idx = 0
    with open(teacher_logits_path, encoding='utf8') as fIn, open(spladecat_train_data_path, 'w', encoding='utf8') as fOut:
        for line in fIn:
            pos_score, neg_score, qid, pid1, pid2 = line.strip().split("\t")
            if qid in dev_qids:
                continue
            line_idx += 1
            if line_idx % SAMPLE_EVERY_N != 0:
                continue
            if qid not in train_queries or pid1 not in train_corpus or pid2 not in train_corpus:
                continue
            if qid not in splade_all_scores or pid1 not in splade_all_scores.get(qid, {}) or pid2 not in splade_all_scores.get(qid, {}):
                continue
            q1 = f"{splade_all_scores[qid][pid1]} [SEP] {train_queries[qid]}"
            q2 = f"{splade_all_scores[qid][pid2]} [SEP] {train_queries[qid]}"
            fOut.write(f"{q1}\t{train_corpus[pid1]}\t{pos_score}\n")
            fOut.write(f"{q2}\t{train_corpus[pid2]}\t{neg_score}\n")
            num_samples += 2
    print(f"Wrote {num_samples} training samples")

del train_corpus, train_queries, splade_all_scores; gc.collect()

# ── Build HF Dataset ──
def gen_spladecat():
    with open(spladecat_train_data_path, 'r', encoding='utf8') as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 3:
                yield {"sentence1": parts[0], "sentence2": parts[1], "label": float(parts[2])}

spladecat_dataset = Dataset.from_generator(gen_spladecat)
print(f"SPLADECAT dataset: {len(spladecat_dataset)} samples")

# ── Train ──
spladecat_model = STCrossEncoder(
    BASE_MODEL, num_labels=1, max_length=512,
    activation_fn=torch.nn.Identity(), device='cuda'
)

spladecat_evaluator = CERerankingEvaluator(dev_samples, name='spladecat-eval')

spladecat_args = CrossEncoderTrainingArguments(
    output_dir=SPLADECAT_SAVE_PATH,
    num_train_epochs=1,
    per_device_train_batch_size=256,
    warmup_steps=625,
    learning_rate=7e-6,
    bf16=True,
    eval_strategy='steps',
    eval_steps=5000,
    save_strategy='steps',
    save_steps=5000,
    load_best_model_at_end=True,
    metric_for_best_model='eval_sequential_score',
    greater_is_better=True,
    max_grad_norm=1.0,
    weight_decay=0.01,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    logging_steps=100,
)

spladecat_trainer = CrossEncoderTrainer(
    model=spladecat_model,
    args=spladecat_args,
    train_dataset=spladecat_dataset,
    loss=CEMSELoss(spladecat_model),
    evaluator=[spladecat_evaluator],
)

print("Training SPLADECAT...")
spladecat_trainer.train()
spladecat_model.save(SPLADECAT_SAVE_PATH + '-latest')
print(f"SPLADECAT model saved to {SPLADECAT_SAVE_PATH}-latest")

del spladecat_model, spladecat_trainer, spladecat_dataset, dev_samples
torch.cuda.empty_cache(); gc.collect()

Loading SPLADE training scores...


Reading collection.tsv: 0it [00:00, ?it/s]

Reading queries.train.tsv: 0it [00:00, ?it/s]

Dev set: 200 queries
Writing 10% SPLADECAT training data...
Wrote 7951856 training samples


Generating train split: 0 examples [00:00, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SPLADECAT model saved to finetuned_CEs/spladecat-20260412_180638-latest


3757

## Cell 15: Cross-Encoder (pre-trained, ref) Re-ranking

In [ ]:
print(f"Loading baseline cross-encoder: {BASELINE_CE_MODEL}")
baseline_model = CrossEncoder(BASELINE_CE_MODEL, num_labels=1, max_length=MAX_LENGTH_BASELINE)

baseline_run = {}
for qid in tqdm.tqdm(top1000_data, desc="Baseline re-ranking"):
    query = queries[qid]
    docs = top1000_data[qid]
    model_input = [[query, passage] for did, passage in docs]
    scores = baseline_model.predict(model_input, batch_size=64)
    baseline_run[qid] = {did: float(s) for (did, _), s in zip(docs, scores)}

ALL_RESULTS['Cross-Encoder (pre-trained, ref)'] = evaluate_run(baseline_run, qrel)
print("Pre-trained reference:", ALL_RESULTS['Cross-Encoder (pre-trained, ref)'])

del baseline_model
torch.cuda.empty_cache()

Loading baseline cross-encoder: cross-encoder/ms-marco-MiniLM-L-12-v2


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Baseline re-ranking:   0%|          | 0/43 [00:00<?, ?it/s]

Pre-trained reference: {'recall.10': np.float64(0.17889466488990755), 'ndcg_cut.10': np.float64(0.7427932986615199), 'map_cut.1000': np.float64(0.497408024331533)}


## Cell 16: Cross-Encoder (10% vanilla) Re-ranking

In [ ]:
vanilla_model_path = VANILLA_SAVE_PATH + '-latest'
print(f"Loading vanilla cross-encoder (10% data): {vanilla_model_path}")
vanilla_ce = CrossEncoder(vanilla_model_path, num_labels=1, max_length=MAX_LENGTH_BASELINE)

vanilla_run = {}
for qid in tqdm.tqdm(top1000_data, desc="Vanilla 10% re-ranking"):
    query = queries[qid]
    docs = top1000_data[qid]
    model_input = [[query, passage] for did, passage in docs]
    scores = vanilla_ce.predict(model_input, batch_size=64)
    vanilla_run[qid] = {did: float(s) for (did, _), s in zip(docs, scores)}

ALL_RESULTS['Cross-Encoder (10% vanilla)'] = evaluate_run(vanilla_run, qrel)
print("Vanilla 10%:", ALL_RESULTS['Cross-Encoder (10% vanilla)'])

del vanilla_ce
torch.cuda.empty_cache()


Loading vanilla cross-encoder (10% data): finetuned_CEs/vanilla-20260412_192316-latest


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Vanilla 10% re-ranking:   0%|          | 0/43 [00:00<?, ?it/s]

Vanilla 10%: {'recall.10': np.float64(0.11203846148832489), 'ndcg_cut.10': np.float64(0.5401274509032217), 'map_cut.1000': np.float64(0.34679002481041965)}


## Cell 17: Cross-Encoder + BM25CAT Re-ranking

In [ ]:
bm25cat_model_path = BM25CAT_SAVE_PATH + '-latest'
print(f"Loading BM25CAT model: {bm25cat_model_path}")
bm25cat_ce = CrossEncoder(bm25cat_model_path, num_labels=1, max_length=MAX_LENGTH_INJECTED)

# Normalize BM25 scores for injection
bm25_norm = normalize_scores(bm25_scores, BM25_MIN, BM25_MAX)

bm25cat_run = {}
for qid in tqdm.tqdm(top1000_data, desc="BM25CAT re-ranking"):
    query = queries[qid]
    docs = top1000_data[qid]
    model_input = [
        [f"{bm25_norm[qid][did]} [SEP] {query}", passage]
        for did, passage in docs
    ]
    scores = bm25cat_ce.predict(model_input, batch_size=64)
    bm25cat_run[qid] = {did: float(s) for (did, _), s in zip(docs, scores)}

ALL_RESULTS['Cross-Encoder + BM25CAT'] = evaluate_run(bm25cat_run, qrel)
print("BM25CAT:", ALL_RESULTS['Cross-Encoder + BM25CAT'])

del bm25cat_ce
torch.cuda.empty_cache()

Loading BM25CAT model: finetuned_CEs/bm25cat-20260412_164802-latest


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BM25CAT re-ranking:   0%|          | 0/43 [00:00<?, ?it/s]

BM25CAT: {'recall.10': np.float64(0.14775036351184015), 'ndcg_cut.10': np.float64(0.6654278901726992), 'map_cut.1000': np.float64(0.43718571297325715)}


## Cell 18: Cross-Encoder + SPLADECAT Re-ranking

In [ ]:
spladecat_model_path = SPLADECAT_SAVE_PATH + '-latest'
print(f"Loading SPLADECAT model: {spladecat_model_path}")
spladecat_ce = CrossEncoder(spladecat_model_path, num_labels=1, max_length=MAX_LENGTH_INJECTED)

# Normalize SPLADE scores for injection
splade_norm = normalize_scores(splade_scores, SPLADE_MIN, SPLADE_MAX)

spladecat_run = {}
for qid in tqdm.tqdm(top1000_data, desc="SPLADECAT re-ranking"):
    query = queries[qid]
    docs = top1000_data[qid]
    model_input = [
        [f"{splade_norm[qid][did]} [SEP] {query}", passage]
        for did, passage in docs
    ]
    scores = spladecat_ce.predict(model_input, batch_size=64)
    spladecat_run[qid] = {did: float(s) for (did, _), s in zip(docs, scores)}

ALL_RESULTS['Cross-Encoder + SPLADECAT'] = evaluate_run(spladecat_run, qrel)
print("SPLADECAT:", ALL_RESULTS['Cross-Encoder + SPLADECAT'])

del spladecat_ce
torch.cuda.empty_cache()

Loading SPLADECAT model: finetuned_CEs/spladecat-20260412_180638-latest


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

SPLADECAT re-ranking:   0%|          | 0/43 [00:00<?, ?it/s]

SPLADECAT: {'recall.10': np.float64(0.17075324004175976), 'ndcg_cut.10': np.float64(0.7240503801735254), 'map_cut.1000': np.float64(0.48172376602316475)}


## Cell 19: Results Comparison Table

In [ ]:
# Build results DataFrame
rows = []
system_order = [
    'BM25 (first-stage)',
    'SPLADE (first-stage)',
    'Cross-Encoder (pre-trained, ref)',
    'Cross-Encoder (10% vanilla)',
    'Cross-Encoder + BM25CAT',
    'Cross-Encoder + SPLADECAT',
]

for system in system_order:
    if system in ALL_RESULTS:
        r = ALL_RESULTS[system]
        rows.append({
            'System': system,
            'nDCG@10': r.get('ndcg_cut.10', 0),
            'MAP@1000': r.get('map_cut.1000', 0),
            'Recall@10': r.get('recall.10', 0),
        })

df = pd.DataFrame(rows)
df = df.set_index('System')

# Format as percentages
print("\n" + "=" * 75)
print("TREC DL'19 — Ablation Comparison")
print("=" * 75)
print(df.to_string(float_format=lambda x: f"{x:.4f}"))
print("=" * 75)

# Key comparisons
if 'Cross-Encoder + SPLADECAT' in ALL_RESULTS and 'SPLADE (first-stage)' in ALL_RESULTS:
    spladecat_ndcg = ALL_RESULTS['Cross-Encoder + SPLADECAT']['ndcg_cut.10']
    splade_ndcg = ALL_RESULTS['SPLADE (first-stage)']['ndcg_cut.10']
    vanilla_ndcg = ALL_RESULTS.get('Cross-Encoder (10% vanilla)', {}).get('ndcg_cut.10', 0)
    bm25cat_ndcg = ALL_RESULTS.get('Cross-Encoder + BM25CAT', {}).get('ndcg_cut.10', 0)
    ref_ndcg = ALL_RESULTS.get('Cross-Encoder (pre-trained, ref)', {}).get('ndcg_cut.10', 0)

    print(f"\nKey Findings:")
    print(f"  Vanilla CE (10%):       {vanilla_ndcg:.4f}")
    print(f"  BM25CAT (10%):          {bm25cat_ndcg:.4f}")
    print(f"  SPLADECAT (10%):        {spladecat_ndcg:.4f}")
    print(f"  SPLADE first-stage:      {splade_ndcg:.4f}")
    print(f"  Pre-trained CE (ref):    {ref_ndcg:.4f}")
    print("  ")
    print(f"  BM25CAT vs vanilla:      +{bm25cat_ndcg - vanilla_ndcg:.4f} ({(bm25cat_ndcg/vanilla_ndcg - 1)*100:+.1f}%)")
    print(f"  SPLADECAT vs vanilla:    +{spladecat_ndcg - vanilla_ndcg:.4f} ({(spladecat_ndcg/vanilla_ndcg - 1)*100:+.1f}%)")
    print(f"  SPLADECAT vs SPLADE:     +{spladecat_ndcg - splade_ndcg:.4f} ({(spladecat_ndcg/splade_ndcg - 1)*100:+.1f}%)")
    print("  ")
    if spladecat_ndcg > splade_ndcg:
        print(f"  --> SPLADECAT still beats SPLADE first-stage once trained on the same 10% slice.")
    else:
        print(f"  --> SPLADE first-stage remains competitive with SPLADECAT under matched training.")


TREC DL'19 — Ablation Comparison
                                  nDCG@10  MAP@1000  Recall@10
System                                                        
BM25 (first-stage)                 0.5203    0.3719     0.1385
SPLADE (first-stage)               0.7308    0.5257     0.1724
Cross-Encoder (pre-trained, ref)   0.7428    0.4974     0.1789
Cross-Encoder (10% vanilla)        0.5401    0.3468     0.1120
Cross-Encoder + BM25CAT            0.6654    0.4372     0.1478
Cross-Encoder + SPLADECAT          0.7241    0.4817     0.1708

Key Findings:
  Vanilla CE (10%):       0.5401
  BM25CAT (10%):          0.6654
  SPLADECAT (10%):        0.7241
  SPLADE first-stage:      0.7308
  Pre-trained CE (ref):    0.7428
  
  BM25CAT vs vanilla:      +0.1253 (+23.2%)
  SPLADECAT vs vanilla:    +0.1839 (+34.1%)
  SPLADECAT vs SPLADE:     +-0.0068 (-0.9%)
  
  --> SPLADE first-stage remains competitive with SPLADECAT under matched training.
